In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.signal import hilbert, butter, filtfilt
from scipy.stats import circmean, circstd, ttest_ind, mannwhitneyu
from scipy.ndimage import gaussian_filter1d
import warnings
warnings.filterwarnings('ignore')

# 1. Global font and axis styling
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
mpl.rcParams['axes.edgecolor'] = 'black'  # Use black axis borders
mpl.rcParams['axes.linewidth'] = 1.2
mpl.rcParams['xtick.color'] = 'black'
mpl.rcParams['ytick.color'] = 'black'

# Point both x-axis and y-axis ticks inward
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.major.width'] = 1.2
mpl.rcParams['ytick.major.width'] = 1.2
mpl.rcParams['xtick.major.size'] = 4.0
mpl.rcParams['ytick.major.size'] = 4.0

plt.rcParams.update({
    'font.size': 8+5,          # Default text
    'axes.titlesize': 8+5,     # Title
    'axes.labelsize': 8+5,     # Axis labels
    'xtick.labelsize': 7+5,    # X-axis tick labels
    'ytick.labelsize': 7+5,    # Y-axis tick labels
    'legend.fontsize': 7+5,    # Legend
    'figure.titlesize': 9+5,   # Figure-level title
})


# 2. Define the muted color palette
color_bio = '#5C8EBC'  # Muted blue
color_hw = '#E58A46'   # Warm orange


In [ ]:
file_path = './data/theta_gamma_bio_vs_hard_results_ok.npz'
# Load the precomputed coupling results
try:
    data = np.load(file_path)
    bio_amp_phases = data['bio_amp_phases']
    hard_amp_phases = data['hard_amp_phases']
    bio_MI, hard_MI = data['bio_MI'], data['hard_MI']
    bio_MVL, hard_MVL = data['bio_MVL'], data['hard_MVL']
    p_value_mi = data['p_value_mi']
    p_value_mvl = data['p_value_mvl']
except FileNotFoundError:
    print("Data file not found. Please verify the file path.")

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(3.2*1.18, 5.4*0.8/2*1.18))

n_bins = 18
phase_bins = np.linspace(-np.pi, np.pi, n_bins + 1)
bin_centers = (phase_bins[:-1] + phase_bins[1:]) / 2
bin_centers = bin_centers / np.pi * 180

if len(bio_amp_phases) > 0 and len(hard_amp_phases) > 0:
    bio_amp_mean = np.mean(bio_amp_phases, axis=0)
    bio_amp_sem = np.std(bio_amp_phases, axis=0) / np.sqrt(len(bio_amp_phases))
    hard_amp_mean = np.mean(hard_amp_phases, axis=0)
    hard_amp_sem = np.std(hard_amp_phases, axis=0) / np.sqrt(len(hard_amp_phases))
    
    bio_amp_mean_norm = bio_amp_mean / np.max(bio_amp_mean)
    hard_amp_mean_norm = hard_amp_mean / np.max(hard_amp_mean)
  
    # Use smooth, marker-free curves with increased line width
    ax1.plot(bin_centers, bio_amp_mean_norm, '-', color=color_bio, linewidth=2.5, label='Bio')
    ax1.plot(bin_centers, hard_amp_mean_norm, '-', color=color_hw, linewidth=2.5, label='BNC')
    
    # Use translucent error bands without outlines
    ax1.fill_between(bin_centers, 
                     bio_amp_mean_norm - bio_amp_sem/np.max(bio_amp_mean),
                     bio_amp_mean_norm + bio_amp_sem/np.max(bio_amp_mean),
                     color=color_bio, alpha=0.25, edgecolor='none')
    ax1.fill_between(bin_centers,
                     hard_amp_mean_norm - hard_amp_sem/np.max(hard_amp_mean),
                     hard_amp_mean_norm + hard_amp_sem/np.max(hard_amp_mean),
                     color=color_hw, alpha=0.25, edgecolor='none')
                     
    ax1.set_xlabel(r'Theta phase ($^\circ$)', color='#333333')
    ax1.set_ylabel('Normalized \ngamma amplitude', color='#333333')
    
    # Use a frameless legend
    ax1.legend(loc='upper right', bbox_to_anchor=(1.2, 1), frameon=False)
    
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('./figures/theta_gamma_phase_amplitude_hist_optimized.svg', dpi=300, bbox_inches='tight')
plt.show()
# ==========================================
# Helper function for significance brackets and labels
# ==========================================
def draw_significance(ax, x1, x2, y, h, p_value):
    if p_value < 0.001: sig_text = '***'
    elif p_value < 0.01: sig_text = '**'
    elif p_value < 0.05: sig_text = '*'
    else: sig_text = 'n.s.'
    
    # Draw the horizontal bracket and its short vertical ends
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.2, c='#555555')
    # Add the significance label
    ax.text((x1+x2)*.5, y+h, sig_text, ha='center', va='bottom', color='#333333')

In [ ]:
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(4.5*0.8*1.37, 2.3*0.8*1.37))

# --- MI plot ---
data_to_plot_mi = [bio_MI, hard_MI]
# Use borderless boxes with dark median, whisker, and cap lines
bp3 = ax3.boxplot(data_to_plot_mi, labels=['Bio', 'BNC'], patch_artist=True, widths=0.5,
                  boxprops=dict(linewidth=0,alpha=0.68),  # Borderless box design
                  medianprops=dict(color='#555555', linewidth=1),  # Dark median line
                  whiskerprops=dict(color='#555555', linewidth=1.2),
                  capprops=dict(color='#555555', linewidth=1.2),
                  showfliers=False)

bp3['boxes'][0].set_facecolor(color_bio)
bp3['boxes'][1].set_facecolor(color_hw)

# Overlay all individual MI values on the boxes
rng = np.random.default_rng(20260801)
for position, values, color in [(1, bio_MI, color_bio), (2, hard_MI, color_hw)]:
    values = np.asarray(values, dtype=float).reshape(-1)
    jitter = rng.uniform(-0.10, 0.10, size=len(values))
    ax3.scatter(position + jitter, values, s=18, color=color, edgecolor='white',
                linewidth=0.35, alpha=0.90, zorder=3)

ax3.set_ylabel('MI', color='#333333')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# Calculate the significance-bracket position dynamically
y_max_mi = max(np.max(bio_MI), np.max(hard_MI))
y_range_mi = y_max_mi - min(np.min(bio_MI), np.min(hard_MI))
draw_significance(ax3, 1, 2, y_max_mi + 0.05*y_range_mi, 0.05*y_range_mi, p_value_mi)


# --- MVL plot ---
data_to_plot_mvl = [bio_MVL, hard_MVL]
bp4 = ax4.boxplot(data_to_plot_mvl, labels=['Bio', 'BNC'], patch_artist=True, widths=0.5,
                  boxprops=dict(linewidth=0,alpha=0.68),
                  medianprops=dict(color='#555555', linewidth=1),
                  whiskerprops=dict(color='#555555', linewidth=1.2),
                  capprops=dict(color='#555555', linewidth=1.2),
                  showfliers=False)

bp4['boxes'][0].set_facecolor(color_bio)
bp4['boxes'][1].set_facecolor(color_hw)

# Overlay all individual MVL values on the boxes
for position, values, color in [(1, bio_MVL, color_bio), (2, hard_MVL, color_hw)]:
    values = np.asarray(values, dtype=float).reshape(-1)
    jitter = rng.uniform(-0.10, 0.10, size=len(values))
    ax4.scatter(position + jitter, values, s=18, color=color, edgecolor='white',
                linewidth=0.35, alpha=0.9, zorder=3)

ax4.set_ylabel('MVL',  color='#333333')
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# Calculate the significance-bracket position dynamically
y_max_mvl = max(np.max(bio_MVL), np.max(hard_MVL))
y_range_mvl = y_max_mvl - min(np.min(bio_MVL), np.min(hard_MVL))
draw_significance(ax4, 1, 2, y_max_mvl + 0.05*y_range_mvl, 0.05*y_range_mvl, p_value_mvl)

# Leave enough space above the axes for significance labels
plt.subplots_adjust(top=0.85) 
plt.tight_layout()
plt.savefig('./figures/theta_gamma_bio_vs_hard_comparison_optimized.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(p_value_mi)
print(p_value_mvl)